<a href="https://colab.research.google.com/github/jleonoras/ollama-remote-gpu/blob/main/Remote_Ollama_GPU_Backend_for_VS_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Remote Ollama GPU Backend for VS Code**

# Step #1. Initialize Environment

In [ ]:
#@markdown <h3>Initialize Environment { display-mode: "form" }

import os, subprocess, sys

# CHECKING RUNTIME TYPE
def check_runtime():
    try:
        # Check if NVIDIA driver is accessible
        subprocess.check_output('nvidia-smi')
        print("✅ T4 GPU DETECTED. Proceeding with optimized setup...")
        return True
    except:
        print("\n" + "!"*40)
        print("❌ GPU NOT DETECTED!")
        print("Please go to: Runtime -> Change runtime type -> Select 'T4 GPU' -> Save.")
        print("Then run this cell again.")
        print("!"*40 + "\n")
        return False

if check_runtime():
    print("📦 Installing dependencies quietly...")

    # Clean install sequence - Hidden outputs for a clean dashboard
    !sudo apt-get update -y &> /dev/null
    !sudo apt-get install -y zstd pciutils &> /dev/null

    # Install Ollama (Showing logs as it's quick and confirms GPU detection)
    print("📥 Installing/Updating Ollama...")
    !curl -fsSL https://ollama.com/install.sh | sh

    # Python API dependencies
    print("🐍 Installing Python libraries...")
    !pip install pyngrok ollama -q

    print("\n" + "="*40)
    print("✅ SYSTEM READY!")
    print("Run the next cell to start the API Tunnel.")
    print("="*40)

# Step #2. Start Ollama API Tunnel

In [ ]:
#@markdown <h3>Start Ollama API Tunnel { display-mode: "form" }

#@markdown > Get your token here: [https://dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_TOKEN = "" #@param {type:"string"}
SELECTED_MODEL = "qwen2.5-coder:7b" #@param ["qwen2.5-coder:7b", "deepseek-coder:6.7b", "llama3.1:8b", "codellama:7b", "mistral"] {allow-input: true}
#@markdown > *Tip: You can also manually type other model names from [ollama.com/library](https://ollama.com/library)*

import os, subprocess, time, requests
from pyngrok import ngrok
from IPython.display import HTML, display

# 1. CLEANUP
!pkill ollama
try:
    ngrok.kill()
except:
    pass

# 2. CONFIGURE ENVIRONMENT
os.environ.update({
    'LD_LIBRARY_PATH': '/usr/lib64-nvidia',
    'OLLAMA_HOST': '0.0.0.0:11434',
    'OLLAMA_ORIGINS': '*',
    'NVIDIA_VISIBLE_DEVICES': 'all'
})

# 3. START OLLAMA
print(f"Starting Ollama with T4 GPU Support...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=os.environ)

# 4. SMART HEALTH CHECK
for i in range(15):
    try:
        if requests.get("http://localhost:11434/").status_code == 200:
            print("✅ Ollama Server is UP!")
            break
    except:
        if i == 14: print("❌ Server timeout.")
        time.sleep(2)

# 5. DYNAMIC PULL
print(f"📥 Pulling model: {SELECTED_MODEL}...")
!ollama pull {SELECTED_MODEL}

# 6. FINAL MINIMALIST UI
try:
    if not NGROK_TOKEN or NGROK_TOKEN == "PASTE_YOUR_TOKEN_HERE":
        raise ValueError("Please provide your Ngrok token.")

    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(11434, proto="http").public_url

    ui_html = f"""
    <div style="font-family: 'Segoe UI', sans-serif; font-size: 13.5px; margin-top: 20px; color: #8b949e;">
        <p style="margin-bottom: 10px;">Copy the link below and paste it into VS Code (Continue/Cline settings):</p>
        <div style="font-family: monospace; font-size: 14px; display: flex; align-items: center; gap: 10px;">
            <b style="color: #58a6ff;">Public URL:</b>
            <a href="{public_url}" target="_blank" style="color: #aff5b4; text-decoration: underline;">{public_url}</a>
            <button onclick="copyToClipboard()" id="copy-btn"
                    style="background: #238636; color: white; border: none; border-radius: 4px; padding: 3px 12px; cursor: pointer; font-size: 12px; font-family: sans-serif;">
                Copy to VS Code
            </button>
        </div>
    </div>
    <script>
    function copyToClipboard() {{
        const el = document.createElement('textarea'); el.value = "{public_url}";
        document.body.appendChild(el); el.select(); document.execCommand('copy'); document.body.removeChild(el);
        const btn = document.getElementById("copy-btn"); btn.innerText = "Copied! ✅"; btn.style.backgroundColor = "#2ea043";
        setTimeout(() => {{ btn.innerText = "Copy to VS Code"; btn.style.backgroundColor = "#238636"; }}, 2000);
    }}
    </script>
    """
    display(HTML(ui_html))
except Exception as e:
    print(f"\n❌ ERROR: {e}")